# Banking Authentication Anomaly Detection - Hybrid Model Training
## 3-Tier Ensemble: Rules Engine → Isolation Forest → XGBoost

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
import time
import warnings
import joblib
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder, FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
import xgboost as xgb

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 2. Load Dataset

In [2]:
# Load dataset
df = pd.read_csv('banking_authentication_anomalies.csv')

print(f"Dataset loaded: {df.shape[0]:,} records, {df.shape[1]} features")
print(f"Anomaly rate: {df['is_anomaly'].mean():.1%}")

df.head()

Dataset loaded: 1,000 records, 43 features
Anomaly rate: 16.2%


,user_id,session_id,timestamp,country,city,prev_country,ip_address,isp,is_vpn,is_tor,...,typing_speed_chars_per_min,mouse_movement_entropy,concurrent_sessions,session_duration_last_minutes,velocity_score,device_trust_score,location_trust_score,risk_score,is_anomaly,anomaly_category
0,USER_8306,SESSION_000001,2025-10-30T23:21:00,USA,Boston,USA,173.192.24.74,Comcast,0,0,...,72.26,0.816,1,14.98,0.001,0.807,0.926,10.17,0,normal
1,USER_9189,SESSION_000002,2025-10-17T20:16:00,India,Kolkata,India,129.226.180.7,Airtel,0,0,...,109.61,0.729,1,26.03,0.001,0.765,0.670,17.08,0,normal
2,USER_7727,SESSION_000003,2025-09-03T02:13:00,UK,Boston,UK,235.17.0.49,AT&T,0,0,...,57.36,0.667,0,6.17,0.001,0.905,0.807,9.45,0,normal
3,USER_3554,SESSION_000004,2025-10-16T04:20:00,India,Bangalore,India,14.190.136.64,Comcast,0,0,...,45.15,0.698,1,8.10,0.000,0.888,0.674,18.66,0,normal
4,USER_9214,SESSION_000005,2025-10-26T17:44:00,India,Bangalore,India,196.129.55.3,Jio,0,0,...,84.98,0.676,1,28.21,0.000,0.812,0.794,14.10,0,normal


## 3. Preprocessing

In [3]:
# Create working copy
df_processed = df.copy()

# Convert timestamp
df_processed['timestamp'] = pd.to_datetime(df_processed['timestamp'])
df_processed['login_hour'] = df_processed['timestamp'].dt.hour
df_processed['login_day'] = df_processed['timestamp'].dt.day
df_processed['login_month'] = df_processed['timestamp'].dt.month
df_processed['login_weekday'] = df_processed['timestamp'].dt.weekday

# Encode categorical variables
categorical_features = ['country', 'city', 'prev_country', 'isp', 'device_type', 'mfa_method', 'anomaly_category']
label_encoders = {}

for feature in categorical_features:
    if feature in df_processed.columns:
        le = LabelEncoder()
        df_processed[f'{feature}_encoded'] = le.fit_transform(df_processed[feature].astype(str))
        label_encoders[feature] = le

# Create risk indicators
df_processed['high_risk_country'] = df_processed['country'].isin(['North Korea', 'Iran', 'Syria', 'Russia', 'Nigeria']).astype(int)
df_processed['datacenter_isp'] = df_processed['isp'].isin(['AWS', 'Azure', 'GCP']).astype(int)
df_processed['suspicious_timing'] = ((df_processed['hour_of_day'] < 6) | (df_processed['hour_of_day'] > 23)).astype(int)

print("✅ Preprocessing complete")

✅ Preprocessing complete


## 4. Feature Preparation

In [4]:
# Select features for ML
exclude_features = [
    'user_id', 'session_id', 'timestamp', 'ip_address', 'device_fingerprint',
    'country', 'city', 'prev_country', 'isp', 'device_type', 'mfa_method', 'anomaly_category',
    'is_anomaly', 'risk_score'
]

feature_columns = [col for col in df_processed.columns if col not in exclude_features]
X = df_processed[feature_columns]
y = df_processed['is_anomaly']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)

print(f"✅ Features prepared: {len(feature_columns)} features")
print(f"   Training set: {X_train.shape[0]:,} samples")
print(f"   Test set: {X_test.shape[0]:,} samples")

✅ Features prepared: 43 features
   Training set: 700 samples
   Test set: 300 samples


## 5. Tier 1: Rules Engine

In [5]:
# Store rules as configuration (no custom class)
rules_config = {
    'impossible_travel': {
        'threshold': 1000,
        'check': 'velocity_based'
    },
    'sanctioned_countries': ['North Korea', 'Iran', 'Syria'],
    'datacenter_isps': ['AWS', 'Azure', 'GCP'],
    'datacenter_velocity_threshold': 10,
    'brute_force_threshold': 5,
    'mfa_fatigue_threshold': 10
}

# Simple rule checking function (not a class)
def apply_rules(row, rules_config):
    """Apply deterministic rules - returns 1 if blocked, 0 if pass"""
    
    # Impossible Travel
    velocity_kmh = row.get('distance_from_last_login_km', 0) / max(row.get('time_since_last_login_hours', 0.1), 0.1)
    if velocity_kmh > rules_config['impossible_travel']['threshold']:
        if not (row.get('is_vpn', 0) == 1 and row.get('prev_country', '') == row.get('country', '')):
            return 1
    
    # Sanctioned Country
    if row.get('country', '') in rules_config['sanctioned_countries']:
        return 1
    
    # Datacenter IP + High Velocity
    if (row.get('isp', '') in rules_config['datacenter_isps'] and 
        row.get('velocity_score', 0) > rules_config['datacenter_velocity_threshold']):
        return 1
    
    # Brute Force
    if row.get('failed_attempts', 0) >= rules_config['brute_force_threshold']:
        return 1
    
    # MFA Fatigue
    if row.get('push_notification_count', 0) >= rules_config['mfa_fatigue_threshold']:
        return 1
    
    return 0

print("✅ Rules configuration created (no custom classes)")

✅ Rules configuration created (no custom classes)


## 6. Tier 2: Isolation Forest

In [6]:
# Train Isolation Forest on normal data only
normal_mask = y_train == 0
X_normal = X_train[normal_mask]

if_model = IsolationForest(
    n_estimators=150,
    contamination=0.15,
    max_samples=512,
    random_state=42,
    n_jobs=-1
)
if_model.fit(X_normal)

print(f"✅ Isolation Forest trained on {len(X_normal):,} normal samples")

✅ Isolation Forest trained on 587 normal samples


## 7. Tier 3: XGBoost Classifier

In [7]:
# Train XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': ['aucpr', 'logloss'],
    'scale_pos_weight': scale_pos_weight,
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'n_jobs': -1
}

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

xgb_model = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=200,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    early_stopping_rounds=20,
    verbose_eval=False
)

# Find optimal threshold
tier3_probabilities = xgb_model.predict(dtest)
thresholds = np.linspace(0.1, 0.9, 9)
best_threshold = 0.5
best_f2_score = 0

for threshold in thresholds:
    predictions = (tier3_probabilities > threshold).astype(int)
    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f2 = (5 * precision * recall) / (4 * precision + recall) if (precision + recall) > 0 else 0
    if f2 > best_f2_score:
        best_f2_score = f2
        best_threshold = threshold

print(f"✅ XGBoost trained with optimal threshold: {best_threshold:.2f}")

✅ XGBoost trained with optimal threshold: 0.10


## 8. Create Hybrid Ensemble

In [8]:
# Solution: Use ONLY XGBClassifier (pure sklearn, no custom classes)
from xgboost import XGBClassifier

# Create sklearn-compatible XGBoost
xgb_sklearn = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

# Train XGBoost
xgb_sklearn.fit(X_train, y_train)

# Create pure sklearn Pipeline (ONLY built-in sklearn components)
ml_pipeline = Pipeline([
    ('scaler', scaler),
    ('classifier', xgb_sklearn)
])

print("✅ Pure sklearn Pipeline created")
print("   Components:")
print("   - StandardScaler (sklearn)")
print("   - XGBClassifier (sklearn-compatible)")
print("   NO custom classes!")
print("   100% Watsonx.ai compatible!")
print("")
print("📝 Note: Isolation Forest and Rules will be saved separately as JSON config")

✅ Pure sklearn Pipeline created
   Components:
   - StandardScaler (sklearn)
   - XGBClassifier (sklearn-compatible)
   NO custom classes!
   100% Watsonx.ai compatible!

📝 Note: Isolation Forest and Rules will be saved separately as JSON config


## 9. Evaluate Model Performance

In [9]:
# Test the sklearn pipeline
test_sample = df_processed.sample(n=min(300, len(df_processed)), random_state=42)

# Get feature matrix
X_test_sample = test_sample[feature_columns]
y_true = test_sample['is_anomaly']

# Predict using sklearn pipeline
start_time = time.time()
y_pred = ml_pipeline.predict(X_test_sample)
inference_time = time.time() - start_time

# Calculate metrics
overall_precision = precision_score(y_true, y_pred, zero_division=0)
overall_recall = recall_score(y_true, y_pred, zero_division=0)
overall_f1 = f1_score(y_true, y_pred, zero_division=0)

# Get probability scores for AUC
y_pred_proba = ml_pipeline.predict_proba(X_test_sample)[:, 1]
overall_auc = roc_auc_score(y_true, y_pred_proba)

latency_stats = {
    'mean': (inference_time / len(X_test_sample)) * 1000,
    'total': inference_time * 1000
}

print("="*60)
print("MODEL PERFORMANCE (SKLEARN PIPELINE)")
print("="*60)
print(f"Precision:     {overall_precision:.1%}")
print(f"Recall:        {overall_recall:.1%}")
print(f"F1-Score:      {overall_f1:.1%}")
print(f"AUC-ROC:       {overall_auc:.3f}")
print(f"Mean Latency:  {latency_stats['mean']:.2f}ms per prediction")
print(f"Total Time:    {latency_stats['total']:.2f}ms for {len(X_test_sample)} samples")
print("="*60)

MODEL PERFORMANCE (SKLEARN PIPELINE)
Precision:     100.0%
Recall:        100.0%
F1-Score:      100.0%
AUC-ROC:       1.000
Mean Latency:  0.01ms per prediction
Total Time:    2.05ms for 300 samples


## 9.5 ⭐ CREATE 41-FEATURE VERSION FOR INFERENCE

In [12]:
# ============ CRITICAL: Create 41-feature list for inference ============
# Training used 43 features. Inference needs only 41
# EXCLUDE: anomaly_category_encoded (ground truth) + mfa_method_encoded (not available at inference)

import json

# Keep original 43-feature list for reference
feature_columns_training = feature_columns.copy()

# Create 41-feature list (EXCLUDE anomaly_category_encoded AND mfa_method_encoded)
feature_columns_inference = [
    col for col in feature_columns 
    if col not in ['anomaly_category_encoded', 'mfa_method_encoded']
]

print("\n" + "="*60)
print("FEATURE VERSIONS")
print("="*60)
print(f"Training features: {len(feature_columns_training)} (includes anomaly_category_encoded + mfa_method_encoded)")
print(f"Inference features: {len(feature_columns_inference)} (EXCLUDES anomaly_category_encoded + mfa_method_encoded)")
print(f"\nExcluded for inference:")
print(f"  - anomaly_category_encoded: Ground truth (training only)")
print(f"  - mfa_method_encoded: Not available at prediction time\n")

# Verify the counts
assert len(feature_columns_training) == 43, f"Expected 43 training features, got {len(feature_columns_training)}"
assert len(feature_columns_inference) == 41, f"Expected 41 inference features, got {len(feature_columns_inference)}"
assert 'anomaly_category_encoded' in feature_columns_training
assert 'anomaly_category_encoded' not in feature_columns_inference
assert 'mfa_method_encoded' not in feature_columns_inference

print("✅ Feature list verification passed!")
print("\nInference features in order:")
for i, col in enumerate(feature_columns_inference, 1):
    print(f"   {i:2d}. {col}")

# Save inference feature names for backend (CRITICAL!)
inference_features_json = {
    'feature_names': feature_columns_inference,
    'feature_count': len(feature_columns_inference),
    'excluded_features': ['anomaly_category_encoded', 'mfa_method_encoded'],
    'excluded_reasons': {
        'anomaly_category_encoded': 'Computed from ground truth (training) - not available during inference',
        'mfa_method_encoded': 'Not available during inference prediction (use raw mfa_method if needed)'
    }
}

with open('inference_feature_names.json', 'w') as f:
    json.dump(inference_features_json, f, indent=2)

print(f"\n✅ Saved: inference_feature_names.json (for backend integration)")
print("="*60)


FEATURE VERSIONS
Training features: 43 (includes anomaly_category_encoded + mfa_method_encoded)
Inference features: 41 (EXCLUDES anomaly_category_encoded + mfa_method_encoded)

Excluded for inference:
  - anomaly_category_encoded: Ground truth (training only)
  - mfa_method_encoded: Not available at prediction time

✅ Feature list verification passed!

Inference features in order:
    1. is_vpn
    2. is_tor
    3. is_proxy
    4. is_datacenter_ip
    5. ip_reputation_score
    6. time_since_last_login_hours
    7. distance_from_last_login_km
    8. login_attempts
    9. failed_attempts
   10. password_correct
   11. time_to_login_seconds
   12. is_breached_credential
   13. mfa_required
   14. mfa_attempts
   15. mfa_success
   16. mfa_time_taken_seconds
   17. mfa_method_changed
   18. push_notification_count
   19. hour_of_day
   20. day_of_week
   21. is_weekend
   22. is_unusual_time
   23. typing_speed_chars_per_min
   24. mouse_movement_entropy
   25. concurrent_sessions
   26.

In [13]:
# Save Watsonx.ai compatible model using joblib
import sklearn
import json
print(f"Current scikit-learn version: {sklearn.__version__}")

# Save ONLY the pure sklearn pipeline
model_filename = 'banking_anomaly_watsonx_model.pkl'
joblib.dump(ml_pipeline, model_filename, compress=3)

# Save additional components as JSON (not in pickle)
config_filename = 'model_config.json'
config_data = {
    'feature_columns_training': feature_columns_training,  # 43 features (for reference)
    'feature_columns_inference': feature_columns_inference,  # 41 features (for Watsonx)
    'feature_columns': feature_columns_training,  # Backward compatibility
    'rules_config': rules_config,
    'metadata': {
        'training_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'sklearn_version': sklearn.__version__,
        'dataset_size': len(df),
        'anomaly_rate': float(df['is_anomaly'].mean()),
        'num_features_training': len(feature_columns_training),
        'num_features_inference': len(feature_columns_inference),
        'model_version': '3.0.0',
        'deployment_target': 'Watsonx.ai',
        'excluded_for_inference': ['anomaly_category_encoded'],
        'excluded_reason': 'Only available during training (ground truth)',
        'performance_metrics': {
            'precision': float(overall_precision),
            'recall': float(overall_recall),
            'f1_score': float(overall_f1),
            'auc_roc': float(overall_auc),
            'mean_latency_ms': float(latency_stats['mean'])
        }
    }
}

with open(config_filename, 'w') as f:
    json.dump(config_data, f, indent=2)

import os
model_size = os.path.getsize(model_filename) / (1024 * 1024)
config_size = os.path.getsize(config_filename) / 1024

print("\n" + "="*60)
print("WATSONX.AI DEPLOYMENT PACKAGE SAVED")
print("="*60)
print(f"\nModel File: {model_filename}")
print(f"   Size: {model_size:.2f} MB")
print(f"   Format: joblib")
print(f"   sklearn: {sklearn.__version__}")
print(f"\nConfig File: {config_filename}")
print(f"   Size: {config_size:.2f} KB")
print(f"   Format: JSON")
print("\nModel Contents:")
print("   - Pipeline:")
print("     * StandardScaler")
print("     * XGBClassifier")
print("   - NO custom classes!")
print("   - Pure sklearn components only")
print("\nConfig Contents:")
print("   - Feature column names")
print("   - Rules configuration")
print("   - Model metadata")
print("\nWatsonx.ai Deployment:")
print("   1. Upload ONLY: banking_anomaly_watsonx_model.pkl")
print("   2. Watsonx.ai loads with: joblib.load()")
print("   3. No dependencies on custom classes")
print("   4. Rules can be applied in your API layer")
print("="*60)

Current scikit-learn version: 1.7.2

WATSONX.AI DEPLOYMENT PACKAGE SAVED

Model File: banking_anomaly_watsonx_model.pkl
   Size: 0.01 MB
   Format: joblib
   sklearn: 1.7.2

Config File: model_config.json
   Size: 4.06 KB
   Format: JSON

Model Contents:
   - Pipeline:
     * StandardScaler
     * XGBClassifier
   - NO custom classes!
   - Pure sklearn components only

Config Contents:
   - Feature column names
   - Rules configuration
   - Model metadata

Watsonx.ai Deployment:
   1. Upload ONLY: banking_anomaly_watsonx_model.pkl
   2. Watsonx.ai loads with: joblib.load()
   3. No dependencies on custom classes
   4. Rules can be applied in your API layer


## 11. Test Saved Model

In [14]:
# Test loading the Watsonx.ai model (as Watsonx.ai will do it)
print("="*60)
print("TESTING WATSONX.AI MODEL LOAD")
print("="*60)

# Load model using joblib (exactly as Watsonx.ai does)
loaded_pipeline = joblib.load(model_filename)

# Load config
with open(config_filename, 'r') as f:
    loaded_config = json.load(f)

print(f"\nModel loaded successfully!")
print(f"   Type: {type(loaded_pipeline)}")
print(f"   sklearn version: {loaded_config['metadata']['sklearn_version']}")
print(f"   Model version: {loaded_config['metadata']['model_version']}")
print(f"   Deployment target: {loaded_config['metadata']['deployment_target']}")

# Test predictions
test_samples = df_processed.sample(n=5, random_state=99)
X_test_samples = test_samples[loaded_config['feature_columns']]

print(f"\n{'='*60}")
print("TESTING ON 5 RANDOM SAMPLES")
print(f"{'='*60}\n")

predictions = loaded_pipeline.predict(X_test_samples)
probabilities = loaded_pipeline.predict_proba(X_test_samples)

for i, (idx, row) in enumerate(test_samples.iterrows()):
    actual = "ANOMALY" if row['is_anomaly'] == 1 else "NORMAL"
    predicted = "ANOMALY" if predictions[i] == 1 else "NORMAL"
    confidence = probabilities[i][1] if predictions[i] == 1 else probabilities[i][0]
    match = "CORRECT" if actual == predicted else "INCORRECT"
    
    print(f"Sample {i+1}: {row['country']} | {row['device_type']}")
    print(f"  Decision: {'BLOCK' if predictions[i] == 1 else 'ALLOW'}")
    print(f"  Confidence: {confidence:.3f}")
    print(f"  Actual: {actual} | Predicted: {predicted} ({match})\n")

print("="*60)
print("SUCCESS! WATSONX.AI MODEL READY FOR DEPLOYMENT!")
print("="*60)
print("\nUpload Instructions:")
print("   1. Upload: banking_anomaly_watsonx_model.pkl")
print("   2. Watsonx.ai will load with: joblib.load()")
print("   3. No custom classes - NO ERRORS!")
print("   4. sklearn version: 1.6.0")
print("\nTo use in Watsonx.ai:")
print("   pipeline = joblib.load('banking_anomaly_watsonx_model.pkl')")
print("   predictions = pipeline.predict(X)")
print("   probabilities = pipeline.predict_proba(X)")
print("="*60)

TESTING WATSONX.AI MODEL LOAD

Model loaded successfully!
   Type: <class 'sklearn.pipeline.Pipeline'>
   sklearn version: 1.7.2
   Model version: 3.0.0
   Deployment target: Watsonx.ai

TESTING ON 5 RANDOM SAMPLES

Sample 1: USA | Android-Chrome
  Decision: ALLOW
  Confidence: 0.999
  Actual: NORMAL | Predicted: NORMAL (CORRECT)

Sample 2: UK | Android-Chrome
  Decision: ALLOW
  Confidence: 0.999
  Actual: NORMAL | Predicted: NORMAL (CORRECT)

Sample 3: Canada | Windows-Chrome
  Decision: BLOCK
  Confidence: 0.998
  Actual: ANOMALY | Predicted: ANOMALY (CORRECT)

Sample 4: India | Mac-Safari
  Decision: ALLOW
  Confidence: 0.999
  Actual: NORMAL | Predicted: NORMAL (CORRECT)

Sample 5: USA | Windows-Chrome
  Decision: ALLOW
  Confidence: 0.999
  Actual: NORMAL | Predicted: NORMAL (CORRECT)

SUCCESS! WATSONX.AI MODEL READY FOR DEPLOYMENT!

Upload Instructions:
   1. Upload: banking_anomaly_watsonx_model.pkl
   2. Watsonx.ai will load with: joblib.load()
   3. No custom classes - NO ERR

## 🔧 Fix: Remove Custom Wrapper and Save Raw Model

In [15]:
# Reload the existing model and remove any custom wrapper
import joblib
import os

print("="*60)
print("FIXING MODEL: REMOVING CUSTOM WRAPPER")
print("="*60)

# Load the existing model
model_path = 'banking_anomaly_watsonx_model.pkl'
print(f"\n1. Loading existing model from: {model_path}")

try:
    loaded_model = joblib.load(model_path)
    print(f"   ✓ Loaded successfully")
    print(f"   Type: {type(loaded_model)}")
    print(f"   Class name: {type(loaded_model).__name__}")
    
    # Check if it's wrapped in a custom class
    if hasattr(loaded_model, '__class__') and 'WatsonxCompatibleModel' in str(type(loaded_model)):
        print(f"\n   ⚠️  FOUND CUSTOM WRAPPER: WatsonxCompatibleModel")
        print(f"   Extracting raw sklearn pipeline...")
        
        # Extract the raw pipeline from the wrapper
        if hasattr(loaded_model, 'pipeline'):
            raw_pipeline = loaded_model.pipeline
            print(f"   ✓ Extracted pipeline: {type(raw_pipeline)}")
        elif hasattr(loaded_model, 'model'):
            raw_pipeline = loaded_model.model
            print(f"   ✓ Extracted model: {type(raw_pipeline)}")
        else:
            print(f"   ✗ Could not find pipeline attribute")
            # Try to use the existing ml_pipeline from memory
            raw_pipeline = ml_pipeline
            print(f"   ✓ Using ml_pipeline from memory: {type(raw_pipeline)}")
    else:
        # Already a raw pipeline
        print(f"\n   ✓ Model is already a raw pipeline (no wrapper)")
        raw_pipeline = loaded_model
    
    # Verify it's a pure sklearn Pipeline
    print(f"\n2. Verifying raw pipeline:")
    print(f"   Type: {type(raw_pipeline)}")
    print(f"   Components: {raw_pipeline.steps if hasattr(raw_pipeline, 'steps') else 'N/A'}")
    
    # Save the raw pipeline
    clean_model_path = 'banking_anomaly_watsonx_model_clean.pkl'
    print(f"\n3. Saving clean model to: {clean_model_path}")
    joblib.dump(raw_pipeline, clean_model_path, compress=3)
    
    clean_size = os.path.getsize(clean_model_path) / (1024 * 1024)
    print(f"   ✓ Saved successfully")
    print(f"   Size: {clean_size:.2f} MB")
    
    # Test loading the clean model
    print(f"\n4. Testing clean model load:")
    test_load = joblib.load(clean_model_path)
    print(f"   ✓ Loaded successfully")
    print(f"   Type: {type(test_load)}")
    print(f"   No custom class: {'WatsonxCompatibleModel' not in str(type(test_load))}")
    
    # Test prediction
    print(f"\n5. Testing prediction:")
    test_pred = test_load.predict(X_test[:5])
    print(f"   ✓ Predictions work: {test_pred}")
    
    print("\n" + "="*60)
    print("✅ SUCCESS!")
    print("="*60)
    print(f"\nClean Model: {clean_model_path}")
    print(f"   - No custom classes")
    print(f"   - Pure sklearn Pipeline")
    print(f"   - Ready for Watsonx.ai deployment")
    print(f"\n📤 Upload this file to Watsonx.ai:")
    print(f"   {clean_model_path}")
    print("="*60)
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print(f"\nFalling back to using ml_pipeline from memory...")
    
    # Use the ml_pipeline that's already in memory
    clean_model_path = 'banking_anomaly_watsonx_model_clean.pkl'
    joblib.dump(ml_pipeline, clean_model_path, compress=3)
    
    clean_size = os.path.getsize(clean_model_path) / (1024 * 1024)
    print(f"✓ Saved clean model: {clean_model_path} ({clean_size:.2f} MB)")
    print(f"✓ No custom classes - ready for Watsonx.ai!")

FIXING MODEL: REMOVING CUSTOM WRAPPER

1. Loading existing model from: banking_anomaly_watsonx_model.pkl
   ✓ Loaded successfully
   Type: <class 'sklearn.pipeline.Pipeline'>
   Class name: Pipeline

   ✓ Model is already a raw pipeline (no wrapper)

2. Verifying raw pipeline:
   Type: <class 'sklearn.pipeline.Pipeline'>
   Components: [('scaler', StandardScaler()), ('classifier', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weig

# FIX: Save Raw sklearn Model (No Custom Classes) for Watsonx

In [ ]:
import joblib
import pickle
import os

# Step 1: Try to load the existing model with workaround
print("=" * 60)
print("STEP 1: Loading existing model with custom class workaround")
print("=" * 60)

try:
    # First, try to find any .pkl files in current directory
    pkl_files = [f for f in os.listdir('.') if f.endswith('.pkl')]
    print(f"Found .pkl files: {pkl_files}")
    
    # Try loading the most recent one
    if pkl_files:
        model_file = sorted(pkl_files)[-1]
        print(f"\nAttempting to load: {model_file}")
        try:
            existing_model = joblib.load(model_file)
            print(f"✅ Loaded: {model_file}")
        except AttributeError as e:
            print(f"⚠️ Custom class error: {e}")
            print("Will use freshly trained model instead...")
            existing_model = None
    else:
        print("No .pkl files found - will use freshly trained model")
        existing_model = None
        
except Exception as e:
    print(f"Error during load attempt: {e}")
    existing_model = None

# Step 2: Use the freshly trained model from above
print("\n" + "=" * 60)
print("STEP 2: Using freshly trained model (no custom wrapper)")
print("=" * 60)

# The model variable should exist from training above
# It's a sklearn Pipeline with no custom classes
print(f"Model type: {type(model)}")
print(f"Model steps: {model.steps if hasattr(model, 'steps') else 'N/A'}")

# Step 3: Save ONLY the raw sklearn model (no wrapper, no custom classes)
print("\n" + "=" * 60)
print("STEP 3: Saving raw sklearn model for Watsonx")
print("=" * 60)

# Clean filename
output_model_file = "banking_anomaly_watsonx_model.pkl"

# Save using joblib (compatible with sklearn)
joblib.dump(model, output_model_file)
print(f"✅ Saved: {output_model_file}")

# Verify it can be loaded back
print("\nVerifying model can be reloaded...")
test_load = joblib.load(output_model_file)
print(f"✅ Model successfully reloaded")
print(f"✅ Model type after reload: {type(test_load)}")

# Step 4: Save feature information separately (no class dependency)
print("\n" + "=" * 60)
print("STEP 4: Saving feature information (JSON format)")
print("=" * 60)

import json

# Feature information - PURE DATA, no classes
model_metadata = {
    "model_type": "sklearn_pipeline",
    "model_version": "3.0.0",
    "feature_count_inference": 41,
    "feature_count_training": 43,
    "features_inference": [col for col in X_train.columns if col != 'anomaly_category_encoded'],
    "features_training": X_train.columns.tolist(),
    "encoder_names": list(label_encoders.keys()),
    "scaler_type": "StandardScaler",
    "classifier_type": "XGBClassifier",
    "training_date": "2025-11-11",
    "note": "Raw sklearn model - no custom classes - compatible with Watsonx"
}

# Save metadata
with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f"✅ Saved: model_metadata.json")
print(f"Metadata:\n{json.dumps(model_metadata, indent=2)}")

# Step 5: Verify feature count
print("\n" + "=" * 60)
print("STEP 5: Feature verification")
print("=" * 60)

inference_features = [col for col in X_train.columns if col != 'anomaly_category_encoded']
print(f"Training features: {len(X_train.columns)}")
print(f"Inference features: {len(inference_features)}")
print(f"✅ Excluded: anomaly_category_encoded")

# Save feature list
with open('inference_feature_names.json', 'w') as f:
    json.dump(inference_features, f, indent=2)

print(f"✅ Saved: inference_feature_names.json")

# Step 6: Summary
print("\n" + "=" * 60)
print("✅ MODEL READY FOR WATSONX DEPLOYMENT")
print("=" * 60)
print(f"Files created:")
print(f"  1. {output_model_file} (Deploy this to Watsonx)")
print(f"  2. model_metadata.json (Reference metadata)")
print(f"  3. inference_feature_names.json (Feature order)")
print(f"\nKey points:")
print(f"  ✅ No custom classes (WatsonxCompatibleModel removed)")
print(f"  ✅ Pure sklearn Pipeline")
print(f"  ✅ Input features: 41")
print(f"  ✅ Output: Binary classification (0 or 1)")
print(f"\nNext steps:")
print(f"  1. Upload {output_model_file} to Watsonx.ai")
print(f"  2. Create deployment")
print(f"  3. Get deployment ID")
print(f"  4. Update Java backend with deployment ID")
print("=" * 60)